In [24]:
import stable_retro as retro
from gymnasium import Env
from gymnasium.spaces import MultiBinary, Box
import numpy as np
import cv2
from matplotlib import pyplot as plt
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import SubprocVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3 import PPO

In [25]:
class StreetFighter(Env):
    def __init__(self):
        super().__init__()
        self.observation_space = Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)
        
        self.action_space = MultiBinary(12)

        self.game = retro.make(game="StreetFighterIISpecialChampionEdition-Genesis-v0", use_restricted_actions=retro.Actions.FILTERED, render_mode=None)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.game.step(action)
        obs = self.preprocess(obs)

        player_matches_won = info["matches_won"]
        enemy_matches_won = info["enemy_matches_won"]
        
        self.previous_frame = obs

        if info["health"] == 0 and info["enemy_health"] == 0:
            reward = 0
            self.enemy_health = 0
            self.player_health = 0
        else:
            dmg_dealt = max(0, self.enemy_health -info["enemy_health"])
            dmg_taken = max(0, self.player_health- info["health"])
            self.enemy_health = info["enemy_health"]
            self.player_health = info["health"]
            reward = dmg_dealt - dmg_taken

        if info["enemy_matches_won"] >= 2 or info["matches_won"] >= 2:
                    terminated = True

        if player_matches_won > self.player_matches_won:
            reward += 200
            if player_matches_won == 2:
                reward += 600
        if enemy_matches_won > self.enemy_matches_won:
            reward -= 200
            if enemy_matches_won == 2:
                reward -= 600

        self.enemy_matches_won = enemy_matches_won
        self.player_matches_won = player_matches_won

        
        return obs, reward, terminated, truncated, info


    def render(self, *args, **kwargs):
        self.game.render()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs, info = self.game.reset(seed=seed, options=options)
        obs = self.preprocess(obs)
        self.previous_frame = obs

        
        info = self.game.data.lookup_all()
        self.player_health = info.get("health", 0)
        self.enemy_health = info.get("enemy_health", 0)
        self.player_matches_won = info.get("matches_won")
        self.enemy_matches_won = info.get("enemy_matches_won")
        return obs, info

    def preprocess(self, observation):
        gray = cv2.cvtColor(observation, cv2.COLOR_RGB2GRAY)

        resize = cv2.resize(gray, (84,84), interpolation=cv2.INTER_AREA)

        channels = np.reshape(resize, (84,84,1))
        return channels


    def close(self):
        self.game.close()

In [ ]:
LOG_DIR = "./opt_logs/"
model_path = "./opt/checkpoints/sf_50m_v4_50000000_steps.zip"
save_path = "./opt/sf_65m_v4"
callback = CheckpointCallback(
    save_freq=25000,
    save_path="./opt/checkpoints/",
    name_prefix="sf_v5"
)
mil_timesteps = 20
timesteps = 1000000 * mil_timesteps

def make_env():
    return Monitor(StreetFighter(), LOG_DIR)

In [ ]:
def main():
    env = SubprocVecEnv([make_env for _ in range(4)])
    env = VecFrameStack(env, n_stack=4, channels_order="last")
    # model = PPO.load(model_path, env, device="mps", verbose=1, tensorboard_log=LOG_DIR, ent_coef=0.001)
    model = PPO("CnnPolicy", env, 
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.001,
        vf_coef=0.5,
        max_grad_norm=0.5,
        target_kl=0.02,
        device="mps",
        verbose=1,
        tensorboard_log="./opt_logs/",
)
    model.learn(total_timesteps=timesteps, progress_bar=True, callback=callback)
    model.save(save_path)
    env.close()
    print("done")

    


if __name__ == "__main__":
    main()

Wrapping the env in a VecTransposeImage.

Logging to ./opt_logs/PPO_9

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 6.64e+03 |
|    ep_rew_mean     | -14      |
| time/              |          |
|    fps             | 1232     |
|    iterations      | 1        |
|    time_elapsed    | 23       |
|    total_timesteps | 28928    |
---------------------------------

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 8.83e+03  |
|    ep_rew_mean          | 6         |
| time/                   |           |
|    fps                  | 756       |
|    iterations           | 2         |
|    time_elapsed         | 76        |
|    total_timesteps      | 57856     |
| train/                  |           |
|    approx_kl            | 0.6085432 |
|    clip_fraction        | 0.0157    |
|    clip_range           | 0.377     |
|    entropy_loss         | -7.82     |
|    explained_variance   | 0.125     |
|    learning_rate        | 6.26e-05  |
|    loss                 | 0.247     |
|    n_updates            | 9300      |
|    policy_gradient_loss | 0.0104    |
|    value_loss           | 11.2      |
---------------------------------------

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 9.81e+03  |
|    ep_rew_mean          | 41.6      |
| time/                   |           |
|    fps                  | 675       |
|    iterations           | 3         |
|    time_elapsed         | 128       |
|    total_timesteps      | 86784     |
| train/                  |           |
|    approx_kl            | 1.2624851 |
|    clip_fraction        | 0.00923   |
|    clip_range           | 0.377     |
|    entropy_loss         | -7.88     |
|    explained_variance   | 0.0806    |
|    learning_rate        | 6.26e-05  |
|    loss                 | 0.0895    |
|    n_updates            | 9304      |
|    policy_gradient_loss | 0.00438   |
|    value_loss           | 10.1      |
---------------------------------------

...

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 1.32e+04  |
|    ep_rew_mean          | 28.9      |
| time/                   |           |
|    fps                  | 319       |
|    iterations           | 517       |
|    time_elapsed         | 46804     |
|    total_timesteps      | 14955776  |
| train/                  |           |
|    approx_kl            | 2.0307908 |
|    clip_fraction        | 0.0142    |
|    clip_range           | 0.377     |
|    entropy_loss         | -7.78     |
|    explained_variance   | 0.195     |
|    learning_rate        | 6.26e-05  |
|    loss                 | 21.7      |
|    n_updates            | 11360     |
|    policy_gradient_loss | 0.0137    |
|    value_loss           | 8.91      |
---------------------------------------

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 1.31e+04  |
|    ep_rew_mean          | 27.5      |
| time/                   |           |
|    fps                  | 319       |
|    iterations           | 518       |
|    time_elapsed         | 46897     |
|    total_timesteps      | 14984704  |
| train/                  |           |
|    approx_kl            | 1.4020483 |
|    clip_fraction        | 0.0121    |
|    clip_range           | 0.377     |
|    entropy_loss         | -7.78     |
|    explained_variance   | 0.149     |
|    learning_rate        | 6.26e-05  |
|    loss                 | 2.13      |
|    n_updates            | 11364     |
|    policy_gradient_loss | 0.00714   |
|    value_loss           | 10.4      |
---------------------------------------

---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 1.33e+04  |
|    ep_rew_mean          | 29.9      |
| time/                   |           |
|    fps                  | 319       |
|    iterations           | 519       |
|    time_elapsed         | 46991     |
|    total_timesteps      | 15013632  |
| train/                  |           |
|    approx_kl            | 1.0418949 |
|    clip_fraction        | 0.0118    |
|    clip_range           | 0.377     |
|    entropy_loss         | -7.83     |
|    explained_variance   | 0.0932    |
|    learning_rate        | 6.26e-05  |
|    loss                 | 14.2      |
|    n_updates            | 11368     |
|    policy_gradient_loss | 0.00729   |
|    value_loss           | 12.2      |
---------------------------------------

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15,013,632/15,000,000  [ 13:03:00 < 0:00:00 , ? it/s ]

done

In [27]:
PPO??

Init signature:
PPO(
    policy: str | type[stable_baselines3.common.policies.ActorCriticPolicy],
    env: gymnasium.core.Env | ForwardRef('VecEnv') | str,
    learning_rate: float | collections.abc.Callable[[float], float] = 0.0003,
    n_steps: int = 2048,
    batch_size: int = 64,
    n_epochs: int = 10,
    gamma: float = 0.99,
    gae_lambda: float = 0.95,
    clip_range: float | collections.abc.Callable[[float], float] = 0.2,
    clip_range_vf: None | float | collections.abc.Callable[[float], float] = None,
    normalize_advantage: bool = True,
    ent_coef: float = 0.0,
    vf_coef: float = 0.5,
    max_grad_norm: float = 0.5,
    use_sde: bool = False,
    sde_sample_freq: int = -1,
    rollout_buffer_class: type[stable_baselines3.common.buffers.RolloutBuffer] | None = None,
    rollout_buffer_kwargs: dict[str, Any] | None = None,
    target_kl: float | None = None,
    stats_window_size: int = 100,
    tensorboard_log: str | None = None,
    policy_kwargs: dict[str, Any] | Non